# Evaluation: FID, SSIM, PSNR, LPIPS, Dice

Computes the metrics from chapter 6.4.2 for one generation run and writes the
results as CSV (one aggregate row + an optional per-sample file for
distribution plots in LaTeX).

Four modes, set via the `MODE` knob below:

- `unconditional`       — random real↔gen pairs; SSIM, PSNR, LPIPS, FID
- `seg2mri`             — exact pairs via `stats.json` cond stems; SSIM, PSNR, LPIPS, FID
- `novel_segmentation`  — generated seg-maps; FID + per-class Dice against a
                           random real seg-map of the same size class
- `pipeline`            — `brain_*.nii.gz` from generate_pipeline.ipynb; random
                           real↔gen pairs against T1N volumes; same metrics
                           as unconditional

Re-run the notebook once per (MODE, MODEL_NAME, SAMPLER) combination. Each
run produces its own CSV in `results/`.

In [ ]:
from pathlib import Path

MODE       = "unconditional"   # "unconditional" | "seg2mri" | "novel_segmentation" | "pipeline"
MODEL_NAME = "baseline"        # any tag — shows up in the CSV and filename
SAMPLER    = "DDIM"
IMAGE_SIZE = 32

# Real reference volumes. For seg2mri / pipeline this is the T1N directory;
# for novel_segmentation it is the seg-mask directory.
REAL_DIR = Path(rf"C:\Users\Sven\Desktop\Data\Processed\brats\{IMAGE_SIZE}")

# Generated samples. For pipeline mode point this at
# conditional/pipeline/<SAMPLER>/<SIZE>/samples and only the brain_*.nii.gz
# files are picked up (tumor_*.nii.gz are skipped via GEN_PATTERN).
GEN_DIR = Path(rf"C:\Users\Sven\Desktop\T3_3200\3DMedDiffusion\Code\unconditional\samples\{IMAGE_SIZE}\{SAMPLER}\samples_{MODEL_NAME}\samples")

# Filter for the generation directory. Default matches every .nii.gz; pipeline
# mode overrides this so only the MRI volumes are evaluated.
GEN_PATTERN = "brain_*.nii.gz" if MODE == "pipeline" else "*.nii.gz"

# stats.json lives next to the samples/ folder — used in seg2mri and
# novel_segmentation modes.
STATS_JSON = GEN_DIR.parent / "stats.json"

# novel_segmentation only — CSV with columns (filename, size_label) that maps
# each real seg-map to its size class.
LABELS_CSV       = Path(r"C:\Users\Sven\Desktop\Data\Processed\brats\seg_size_labels.csv")
NUM_SEG_CLASSES  = 4   # BraTS: 0=background, 1..3=tumor sub-regions

RESULTS_DIR = Path.cwd() / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print(f"mode={MODE}  model={MODEL_NAME}  sampler={SAMPLER}  size={IMAGE_SIZE}")
print(f"real: {REAL_DIR}")
print(f"gen:  {GEN_DIR}  pattern={GEN_PATTERN}")

In [ ]:
import sys

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd()))

import volumes as vol
import metrics as mtr

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

In [ ]:
real_files = vol.list_volumes(REAL_DIR)
gen_files  = vol.list_volumes(GEN_DIR, GEN_PATTERN)
print(f"found {len(real_files)} real, {len(gen_files)} generated")

if MODE == "seg2mri":
    pairs = vol.pair_by_stats(REAL_DIR, gen_files, STATS_JSON)
elif MODE == "novel_segmentation":
    pairs = vol.pair_random_by_class(real_files, LABELS_CSV, gen_files, STATS_JSON, seed=42)
else:
    # unconditional + pipeline both use a random real partner per generated sample
    pairs = vol.pair_random(real_files, gen_files, seed=42)

print(f"prepared {len(pairs)} pairs")

In [ ]:
# Load every volume that's needed into memory once. For 1024 paired volumes
# at 32³ this is ~135 MB — well within limits and saves re-reading from disk
# between the per-pair metrics and FID.
real_paired = [vol.load_volume(r) for r, _ in tqdm(pairs, desc="load real (paired)")]
gen_paired  = [vol.load_volume(g) for _, g in tqdm(pairs, desc="load gen")]

# FID benefits from the full real set when more is available. For seg2mri the
# pairing is already exhaustive (one-to-one), so reuse the paired list.
if MODE == "seg2mri" or len(real_files) <= len(pairs):
    real_all = real_paired
else:
    real_all = [vol.load_volume(f) for f in tqdm(real_files, desc="load real (full)")]

print(f"in memory: {len(real_paired)} paired real, {len(gen_paired)} gen, {len(real_all)} reals for FID")

In [ ]:
# Per-pair metrics. For MRI modes (unconditional, seg2mri) this is SSIM, PSNR
# and LPIPS. For novel_segmentation it is Dice — the generated volumes are
# discrete-label maps where pixel/feature similarity makes no sense.
per_sample_rows = []

if MODE == "novel_segmentation":
    for i, (real, gen) in enumerate(tqdm(list(zip(real_paired, gen_paired)), desc="dice")):
        per_class = mtr.dice_per_class(real, gen, num_classes=NUM_SEG_CLASSES)
        row = {"sample_id": i + 1, "dice_mean": float(np.nanmean(per_class))}
        for k, v in enumerate(per_class, start=1):
            row[f"dice_class_{k}"] = v
        per_sample_rows.append(row)
else:
    lpips_model = mtr.load_lpips(device)
    for i, (real, gen) in enumerate(tqdm(list(zip(real_paired, gen_paired)), desc="ssim/psnr/lpips")):
        per_sample_rows.append({
            "sample_id": i + 1,
            "ssim":  mtr.ssim(real, gen),
            "psnr":  mtr.psnr(real, gen),
            "lpips": mtr.lpips_score(real, gen, lpips_model, device),
        })
    del lpips_model
    if device == "cuda":
        torch.cuda.empty_cache()

per_sample_df = pd.DataFrame(per_sample_rows)
print(per_sample_df.describe())

In [ ]:
inception = mtr.load_inception(device)
fid_score = mtr.fid(real_all, gen_paired, inception, device)

del inception
if device == "cuda":
    torch.cuda.empty_cache()

print(f"FID = {fid_score:.4f}  (over {len(real_all)} real vs {len(gen_paired)} generated)")

In [ ]:
tag = f"{MODE}_{MODEL_NAME}_{SAMPLER}_{IMAGE_SIZE}"

# Per-sample CSV — useful for boxplots / distribution figures in LaTeX.
per_sample_path = RESULTS_DIR / f"per_sample_{tag}.csv"
per_sample_df.to_csv(per_sample_path, index=False, float_format="%.6f")
print(f"wrote {per_sample_path}")

# Aggregate row — one CSV per run, easy to concatenate later in LaTeX.
agg = {
    "mode":       MODE,
    "model":      MODEL_NAME,
    "sampler":    SAMPLER,
    "image_size": IMAGE_SIZE,
    "n_pairs":    len(pairs),
    "n_real_fid": len(real_all),
    "fid":        fid_score,
}

if MODE == "novel_segmentation":
    agg["dice_mean"] = float(per_sample_df["dice_mean"].mean(skipna=True))
    agg["dice_std"]  = float(per_sample_df["dice_mean"].std(skipna=True))
    for k in range(1, NUM_SEG_CLASSES):
        col = f"dice_class_{k}"
        agg[f"{col}_mean"] = float(per_sample_df[col].mean(skipna=True))
        agg[f"{col}_std"]  = float(per_sample_df[col].std(skipna=True))
else:
    for col in ("ssim", "psnr", "lpips"):
        agg[f"{col}_mean"] = float(per_sample_df[col].mean())
        agg[f"{col}_std"]  = float(per_sample_df[col].std())

agg_path = RESULTS_DIR / f"metrics_{tag}.csv"
pd.DataFrame([agg]).to_csv(agg_path, index=False, float_format="%.6f")
print(f"wrote {agg_path}")

In [ ]:
# Quick sanity check — not for the thesis, just to see at a glance whether
# the run looks reasonable before moving on to the next model.
print(f"\n=== {tag} ===")
for k, v in agg.items():
    if isinstance(v, float):
        print(f"  {k:<18} {v:.4f}")
    else:
        print(f"  {k:<18} {v}")